In [11]:
import sys
sys.path.append('курсовая_2')

In [12]:
from __future__ import annotations

import glob
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from dtw_similarity import (
    build_daily_sequences_from_hourly_klines,
    compute_similarity_matrix,
)

from market_graph import build_market_weights
from pair_search import find_multiple_pairs

In [ ]:
# Loading helpers

def utc_day(ts_ms: int) -> int:
    dt = datetime.fromtimestamp(ts_ms / 1000.0, tz=timezone.utc)
    return int(dt.strftime("%Y%m%d"))


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_book_tickers(path: Path) -> dict[str, tuple[float, float]]:
    """
    Supports common cached formats:
      {"BTCUSDT": [bid, ask], ...}
      {"BTCUSDT": {"bid": ..., "ask": ...}, ...}
      {"BTCUSDT": {"bidPrice": ..., "askPrice": ...}, ...}
      [{"symbol": "BTCUSDT", "bidPrice": "...", "askPrice": "..."}, ...]
    """
    data = load_json(path)

    out: dict[str, tuple[float, float]] = {}

    if isinstance(data, list):
        for row in data:
            sym = str(row["symbol"])
            bid = float(row.get("bid", row.get("bidPrice")))
            ask = float(row.get("ask", row.get("askPrice")))
            out[sym] = (bid, ask)
        return out

    if isinstance(data, dict):
        for sym, value in data.items():
            if isinstance(value, (list, tuple)) and len(value) >= 2:
                bid, ask = float(value[0]), float(value[1])
            elif isinstance(value, dict):
                bid = float(value.get("bid", value.get("bidPrice")))
                ask = float(value.get("ask", value.get("askPrice")))
            else:
                raise ValueError(f"Unsupported ticker format for {sym}: {value}")

            out[str(sym)] = (bid, ask)

        return out

    raise ValueError(f"Unsupported book_tickers.json format: {type(data)}")


def discover_kline_files(data_dir: Path) -> dict[str, Path]:
    files = sorted(data_dir.glob("klines_*_1h_240.json"))

    out: dict[str, Path] = {}

    for path in files:
        name = path.name
        prefix = "klines_"
        suffix = "_1h_240.json"

        if not name.startswith(prefix) or not name.endswith(suffix):
            continue

        sym = name[len(prefix):-len(suffix)]
        out[sym] = path

    if not out:
        raise FileNotFoundError("No klines_*_1h_240.json files found")

    return out


def latest_day_base_open(klines_1h: list[list]) -> tuple[int, float]:
    """
    Use the first hourly open of the latest UTC day as base_open.
    This matches the normalization idea in the paper/code.
    """
    rows = []
    for k in klines_1h:
        open_time = int(k[0])
        open_price = float(k[1])
        day = utc_day(open_time)
        rows.append((day, open_time, open_price))

    if not rows:
        raise ValueError("Empty klines")

    latest_day = max(day for day, _, _ in rows)
    latest_rows = [(t, o) for day, t, o in rows if day == latest_day]
    latest_rows.sort(key=lambda x: x[0])

    base_open = latest_rows[0][1]

    if base_open <= 0:
        raise ValueError(f"Invalid base_open={base_open}")

    return latest_day, base_open

# Result helpers

def solutions_to_frame(pairs, symbols: list[str], label: str) -> pd.DataFrame:
    rows = []

    for rank, sol in enumerate(pairs, start=1):
        path_symbols = [symbols[node - 1] for node in sol.path_nodes]

        rows.append({
            "model": label,
            "rank": rank,
            "short_node": sol.short_node,
            "long_node": sol.long_node,
            "short_symbol": symbols[sol.short_node - 1],
            "long_symbol": symbols[sol.long_node - 1],
            "path_nodes": " -> ".join(map(str, sol.path_nodes)),
            "path_symbols": " -> ".join(path_symbols),
            "path_length_edges": max(0, len(sol.path_nodes) - 1),
            "path_weight": sol.path_weight,
            "cycle_nodes": " -> ".join(map(str, sol.cycle_nodes)),
        })

    return pd.DataFrame(rows)


def top_negative_edges_frame(
    w: np.ndarray,
    symbols: list[str],
    label: str,
    top_k: int = 30,
) -> pd.DataFrame:
    rows = []
    n = len(symbols)

    for i in range(1, n + 1):
        for j in range(1, n + 1):
            if i == j:
                continue

            rows.append({
                "model": label,
                "short_node": i,
                "long_node": j,
                "short_symbol": symbols[i - 1],
                "long_symbol": symbols[j - 1],
                "edge_weight": float(w[i, j]),
            })

    df = pd.DataFrame(rows)
    return df.sort_values("edge_weight", ascending=True).head(top_k)


In [ ]:
# Parameters


DATA_DIR = Path("binance_data_cached_20260609")
OUT_DIR = Path("results_fee_cpu_20260609")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TAKER_FEE = 0.001

THRESHOLD = 0.0

MAX_PAIRS = 10
SB_VARIANT = "BSB"
N_RUNS_PER_PAIR = 100
N_ITER = 500
DT = 0.5
SEED = 42
MC = 50
MP = 0.5

DAYS_BACK = 5
EXCLUDE_LAST_DAY = True

In [18]:
book_path = DATA_DIR / "book_tickers.json"
if not book_path.exists():
    raise FileNotFoundError("book_tickers.json not found")

book = load_book_tickers(book_path)
kline_files = discover_kline_files(DATA_DIR)

# Use only symbols that have both ticker and kline data.
symbols = [sym for sym in sorted(kline_files) if sym in book]

if len(symbols) < 3:
    raise ValueError(
        f"Need at least 3 symbols with both book ticker and klines, got {len(symbols)}"
    )

print(f"Loaded symbols: {len(symbols)}")
print(symbols)

daily_seq = {}
base_open = {}
latest_days = {}

for sym in symbols:
    klines = load_json(kline_files[sym])

    daily_seq[sym] = build_daily_sequences_from_hourly_klines(klines)

    latest_day, base = latest_day_base_open(klines)
    latest_days[sym] = latest_day
    base_open[sym] = base

sim, used_days = compute_similarity_matrix(
    symbols=symbols,
    daily_seq=daily_seq,
    days_back=DAYS_BACK,
    exclude_last_day=EXCLUDE_LAST_DAY,
)

bid_raw = np.array([book[sym][0] for sym in symbols], dtype=float)
ask_raw = np.array([book[sym][1] for sym in symbols], dtype=float)
base = np.array([base_open[sym] for sym in symbols], dtype=float)

bid_norm = bid_raw / base
ask_norm = ask_raw / base

# Baseline, without explicit exchange fee.
w_gross = build_market_weights(
    ask=ask_norm,
    bid=bid_norm,
    sim=sim,
    fee_rate=0.0,
)

# Fee-adjusted graph.
w_taker = build_market_weights(
    ask=ask_norm,
    bid=bid_norm,
    sim=sim,
    fee_rate=TAKER_FEE,
)

# Save diagnostics.
np.save(OUT_DIR / "real_market_w_gross_recomputed.npy", w_gross)
np.save(OUT_DIR / "real_market_w_taker_fee.npy", w_taker)
np.save(OUT_DIR / "real_market_similarity_recomputed.npy", sim)



meta = {
    "symbols": symbols,
    "used_days_for_similarity": used_days,
    "latest_days": latest_days,
    "base_open": base_open,
    "taker_fee": TAKER_FEE,
    "threshold": THRESHOLD,
    "max_pairs": MAX_PAIRS,
    "sb_variant": SB_VARIANT,
    "n_runs_per_pair": N_RUNS_PER_PAIR,
    "n_iter": N_ITER,
    "dt": DT,
    "seed": SEED,
}

with open(OUT_DIR / "real_market_fee_experiment_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

top_gross = top_negative_edges_frame(w_gross, symbols, "gross")
top_taker = top_negative_edges_frame(w_taker, symbols, "taker_fee")

top_edges = pd.concat([top_gross, top_taker], ignore_index=True)
top_edges.to_csv(OUT_DIR / "real_market_top_negative_edges_fee_compare.csv", index=False)

print("\nRunning gross model...")
pairs_gross = find_multiple_pairs(
    w=w_gross,
    threshold=THRESHOLD,
    max_pairs=MAX_PAIRS,
    sb_variant=SB_VARIANT,
    n_runs_per_pair=N_RUNS_PER_PAIR,
    n_iter=N_ITER,
    dt=DT,
    mp=0.5,
    mc=50,
    seed=SEED,
    debug=True,
)

print("\nRunning taker-fee model...")
pairs_taker = find_multiple_pairs(
    w=w_taker,
    threshold=THRESHOLD,
    max_pairs=MAX_PAIRS,
    sb_variant=SB_VARIANT,
    n_runs_per_pair=N_RUNS_PER_PAIR,
    n_iter=N_ITER,
    dt=DT,
    mp=0.5,
    mc=50,
    seed=SEED,
    debug=True,
)

df_gross = solutions_to_frame(pairs_gross, symbols, "gross")
df_taker = solutions_to_frame(pairs_taker, symbols, "taker_fee")

df_pairs = pd.concat([df_gross, df_taker], ignore_index=True)

df_pairs_unique = (
    df_pairs
    .sort_values(["model", "path_weight"], ascending=[True, True])
    .drop_duplicates(["model", "short_symbol", "long_symbol"], keep="first")
)

df_pairs_unique.to_csv(OUT_DIR / "real_market_pairs_fee_compare.csv", index=False)



summary = pd.DataFrame([
    {
        "model": "gross",
        "n_pairs": len(pairs_gross),
        "best_weight": float(df_gross["path_weight"].min()) if len(df_gross) else np.nan,
        "mean_weight": float(df_gross["path_weight"].mean()) if len(df_gross) else np.nan,
        "mean_path_length_edges": float(df_gross["path_length_edges"].mean()) if len(df_gross) else np.nan,
    },
    {
        "model": "taker_fee",
        "n_pairs": len(pairs_taker),
        "best_weight": float(df_taker["path_weight"].min()) if len(df_taker) else np.nan,
        "mean_weight": float(df_taker["path_weight"].mean()) if len(df_taker) else np.nan,
        "mean_path_length_edges": float(df_taker["path_length_edges"].mean()) if len(df_taker) else np.nan,
    },
])

summary.to_csv(OUT_DIR / "real_market_fee_compare_summary.csv", index=False)

print("\nSummary:")
print(summary)

print("\nSaved files:")
print("  real_market_pairs_fee_compare.csv")
print("  real_market_fee_compare_summary.csv")
print("  real_market_top_negative_edges_fee_compare.csv")
print("  real_market_w_gross_recomputed.npy")
print("  real_market_w_taker_fee.npy")
print("  real_market_similarity_recomputed.npy")
print("  real_market_fee_experiment_meta.json")

Loaded symbols: 15
['ADAUSDT', 'ATOMUSDT', 'AVAXUSDT', 'BCHUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'DOTUSDT', 'ETCUSDT', 'ETHUSDT', 'LINKUSDT', 'LTCUSDT', 'SOLUSDT', 'TRXUSDT', 'XRPUSDT']

Running gross model...
[k=0] best valid cycle weight=-0.05517447704952697
[k=1] best valid cycle weight=-0.05196616381626175
[k=2] best valid cycle weight=-0.05309065634270321
[k=3] best valid cycle weight=-0.05334113616876185
[k=4] best valid cycle weight=-0.05139084245082098
[k=5] best valid cycle weight=-0.05196431554055646
[k=6] best valid cycle weight=-0.05391492144731829
[k=7] best valid cycle weight=-0.04836443785280059
[k=8] no valid dummy-cycle found

Running taker-fee model...
[k=0] best valid cycle weight=-0.043607049335599904
[k=1] best valid cycle weight=-0.043223489778231525
[k=2] best valid cycle weight=-0.0426118772975484
[k=3] best valid cycle weight=-0.04166965592211848
[k=4] best valid cycle weight=-0.04141926981332626
[k=5] best valid cycle weight=-0.04083874618962019
[k=6] best 

In [21]:
df_pairs_unique = (
    df_pairs
    .sort_values(["model", "path_weight"], ascending=[True, True])
    .drop_duplicates(["model", "short_symbol", "long_symbol"], keep="first").reset_index(drop=True)
)

In [22]:
df_pairs_unique

,model,rank,short_node,long_node,short_symbol,long_symbol,path_nodes,path_symbols,path_length_edges,path_weight,cycle_nodes
0,gross,1,2,10,ATOMUSDT,ETHUSDT,2 -> 13 -> 4 -> 12 -> 6 -> 1 -> 3 -> 7 -> 10,ATOMUSDT -> SOLUSDT -> BCHUSDT -> LTCUSDT -> B...,8,-0.055174,0 -> 2 -> 13 -> 4 -> 12 -> 6 -> 1 -> 3 -> 7 ->...
1,gross,7,2,3,ATOMUSDT,AVAXUSDT,2 -> 13 -> 14 -> 5 -> 7 -> 10 -> 4 -> 12 -> 6 ...,ATOMUSDT -> SOLUSDT -> TRXUSDT -> BNBUSDT -> D...,10,-0.053915,0 -> 2 -> 13 -> 14 -> 5 -> 7 -> 10 -> 4 -> 12 ...
2,gross,4,2,6,ATOMUSDT,BTCUSDT,2 -> 13 -> 14 -> 9 -> 7 -> 10 -> 4 -> 12 -> 6,ATOMUSDT -> SOLUSDT -> TRXUSDT -> ETCUSDT -> D...,8,-0.053341,0 -> 2 -> 13 -> 14 -> 9 -> 7 -> 10 -> 4 -> 12 ...
3,gross,3,12,13,LTCUSDT,SOLUSDT,12 -> 6 -> 1 -> 5 -> 15 -> 4 -> 2 -> 13,LTCUSDT -> BTCUSDT -> ADAUSDT -> BNBUSDT -> XR...,7,-0.053091,0 -> 12 -> 6 -> 1 -> 5 -> 15 -> 4 -> 2 -> 13 -> 0
4,gross,2,2,15,ATOMUSDT,XRPUSDT,2 -> 7 -> 10 -> 4 -> 12 -> 15,ATOMUSDT -> DOGEUSDT -> ETHUSDT -> BCHUSDT -> ...,5,-0.051966,0 -> 2 -> 7 -> 10 -> 4 -> 12 -> 15 -> 0
5,gross,6,2,13,ATOMUSDT,SOLUSDT,2 -> 7 -> 10 -> 4 -> 12 -> 13,ATOMUSDT -> DOGEUSDT -> ETHUSDT -> BCHUSDT -> ...,5,-0.051964,0 -> 2 -> 7 -> 10 -> 4 -> 12 -> 13 -> 0
6,gross,5,12,10,LTCUSDT,ETHUSDT,12 -> 6 -> 1 -> 7 -> 13 -> 4 -> 2 -> 10,LTCUSDT -> BTCUSDT -> ADAUSDT -> DOGEUSDT -> S...,7,-0.051391,0 -> 12 -> 6 -> 1 -> 7 -> 13 -> 4 -> 2 -> 10 -> 0
7,gross,8,12,11,LTCUSDT,LINKUSDT,12 -> 6 -> 1 -> 5 -> 7 -> 10 -> 4 -> 2 -> 11,LTCUSDT -> BTCUSDT -> ADAUSDT -> BNBUSDT -> DO...,8,-0.048364,0 -> 12 -> 6 -> 1 -> 5 -> 7 -> 10 -> 4 -> 2 ->...
8,taker_fee,1,2,15,ATOMUSDT,XRPUSDT,2 -> 13 -> 4 -> 12 -> 15,ATOMUSDT -> SOLUSDT -> BCHUSDT -> LTCUSDT -> X...,4,-0.043607,0 -> 2 -> 13 -> 4 -> 12 -> 15 -> 0
9,taker_fee,2,2,13,ATOMUSDT,SOLUSDT,2 -> 15 -> 4 -> 12 -> 13,ATOMUSDT -> XRPUSDT -> BCHUSDT -> LTCUSDT -> S...,4,-0.043223,0 -> 2 -> 15 -> 4 -> 12 -> 13 -> 0


In [25]:
df_pairs[df_pairs["model"] == "gross"]["path_length_edges"].mean()

np.float64(7.25)